## Notebook grammar

Open in Colab: https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne-sanity-delta-test-hierarchical-global-local-oddball.ipynb

setup -> config -> simulation/probe -> objective cells -> export

This notebook uses package APIs through `import jaxfne as jtfne`; editable inputs are centralized in config cells; readouts are proxy-scoped where named as proxies; exports use JSON/PNG receipts when artifacts are produced.


# Hierarchical Global-Local Oddball: Sanity Delta v0.3.32


```python
import jaxfne as jtfne
```
Package-native orchestration for 5-area hierarchical network with AAAB stimulus oddball.

**Status:** Package API hardening. Evidence artifacts for this notebook belong to v0.3.33 hierarchical-suite release.

**Truth gates:**
- ``
- `computational_scaffold`
- `linear_solver`
- `physical_amplitude_calibrated=False`
- `biological_learning_claim=False`

## 1. Imports and Setup

In [ ]:
import jaxfne as jtfne
import jax
import jax.numpy as jnp
import json
from pathlib import Path
from datetime import datetime, timezone

jtfne.enable_x64()
print(f"JAX devices: {jax.devices()}")
print(f"jaxfne version: {jtfne.__version__}")

## 2. Configuration and Model Construction

In [ ]:
# Create hierarchical global-local oddball configuration
cfg = jtfne.SanityDeltaConfig.hierarchical_global_local_oddball()

print(f"Areas: {cfg.areas}")
print(f"Total neurons: {len(cfg.areas) * cfg.neurons_per_area}")
print(f"Cell counts: {cfg.cell_counts}")
print(f"\nTruth gates:")
print(f"  claim_level: {cfg.claim_level}")
print(f"  field_solver_status: {cfg.field_solver_status}")
print(f"  physical_amplitude_calibrated: {cfg.physical_amplitude_calibrated}")
print(f"  biological_learning_claim: {cfg.biological_learning_claim}")

## 3. Paradigm and Task Schedule

In [ ]:
# Create task paradigm
paradigm = cfg.make_paradigm()

print(f"Paradigm: {paradigm.name}")
print(f"Sequence: {paradigm.sequence}")

# Compile schedule
schedule = paradigm.to_schedule()

print(f"\nTotal duration: {schedule['total_duration_ms']} ms")
print(f"Number of segments: {len(schedule['segments'])}")
print(f"\nFirst 3 segments:")
for seg in schedule['segments'][:3]:
    print(f"  {seg['segment_id']}: {seg['segment_type']} ({seg['duration_ms']} ms)")

## 4. Model Construction and Plasticity

In [ ]:
# Construct model
model = cfg.construct()

print(f"Model neurons: {model.n_neurons}")
print(f"Plasticity enabled: {model.plasticity_enabled}")

# Enable plasticity (computational regularizer, not biological learning)
model_with_plasticity = model.enable_plasticity(
    homeostatic_rule="low_rate_potentiation_high_rate_depression",
    target_rate_hz=10.0,
    weight_bounds="preserve_sign",
)

print(f"\nWith plasticity:")
print(f"  Enabled: {model_with_plasticity.plasticity_enabled}")
print(f"  Rule: {model_with_plasticity.plasticity_config['homeostatic_rule']}")
print(f"  Target rate: {model_with_plasticity.plasticity_config['target_rate_hz']} Hz")

## 5. Fixation Gate Setup

In [ ]:
# Create fixation gate for PFC superficial activity
gate = paradigm.make_fixation_gate(
    area="PFC",
    layer_group="superficial",
    target_rate_hz=10.0,
    tolerance_hz=1.0,
    window_ms=500.0,
)

print(f"Fixation gate:")
print(f"  Area: {gate.area}")
print(f"  Layer group: {gate.layer_group}")
print(f"  Target rate: {gate.target_rate_hz} ± {gate.tolerance_hz} Hz")
print(f"  Window: {gate.window_ms} ms")

## 6. Backup State and Task Execution

In [ ]:
# Initialize backup state
backup = model.initialize_backup(paradigm)

print(f"Backup state:")
print(f"  Time: {backup.time_ms} ms")
print(f"  Fixation counter: {backup.fixation_counter}")
print(f"  Ring buffer size: {backup.runtime_metadata.get('ring_buffer_ms', 'not set')} ms")
print(f"  Neurons: {backup.runtime_metadata['n_neurons']}")

## 7. Run Task Episode

In [ ]:
# Run task episode (uses model_with_plasticity for full featured run)
episode = model_with_plasticity.run_task(
    paradigm=paradigm,
    gate=gate,
    backup=backup,
)

print(f"Episode completed:")
print(f"  Spikes shape: {episode.spikes.shape}")
print(f"  Vm shape: {episode.vm.shape}")
print(f"  Time steps: {len(episode.t_ms)}")

## 8. Probing and Validation

In [ ]:
# Probe with proxy readouts
episode.probe(readouts=("spk", "vm", "lfp_proxy", "csd_proxy", "eeg_proxy", "meg_proxy"))

# Validate truth gates
validation = episode.validate(checks=(
    "truth_gates_preserved",
    "finite_outputs",
    "proxy_safe_readouts",
))

print(f"Validation results:")
for check, result in validation.items():
    print(f"  {check}: {result}")

## 9. Manifest and Output

In [ ]:
# Create manifest for reproducibility
manifest = jtfne.Manifest(
    config=cfg,
    paradigm=paradigm,
    backup=backup,
    episode_metadata={
        "duration_ms": cfg.duration_ms,
        "dt_ms": cfg.dt_ms,
        "n_neurons": len(cfg.areas) * cfg.neurons_per_area,
        "areas": list(cfg.areas),
        "stimulus_map": cfg.stimulus_map,
    },
    generated_at_utc=datetime.now(timezone.utc).isoformat(),
)

print(f"Manifest created:")
d = manifest.to_dict()
print(f"  claim_level: {d['claim_level']}")
print(f"  total_neurons: {d['total_neurons']}")

## 10. Summary

**v0.3.32 API Status: Complete**

- ✓ 5-area hierarchical configuration (V1a, V1b, V4, MT, PFC)
- ✓ 500 neurons total (100 per area, 70E + 30I)
- ✓ AAAB oddball stimulus sequence
- ✓ Fixation gate on PFC superficial activity (10±1 Hz)
- ✓ Backup/resume with 1000 ms ring buffer
- ✓ Plasticity: computational homeostatic regularizer
- ✓ Proxy-safe readouts: lfp_proxy, csd_proxy, eeg_proxy, meg_proxy
- ✓ Truth gates preserved: ``, `computational_scaffold`, `linear_solver`

Evidence artifacts for this scaffold belong to **v0.3.33 hierarchical-suite release**.